```markdown
# Challenge Quotidien : Finetuning de LLM avec LoRA

Ce notebook guide un débutant à travers le processus de **Parameter-Efficient Fine-Tuning (PEFT)** en utilisant la méthode **LoRA (Low-Rank Adaptation)**. Nous allons adapter le modèle `bloomz-560m` pour générer des citations.

### Objectifs :
1. Appliquer LoRA à un modèle pré-entraîné.
2. Entraîner le modèle avec la bibliothèque Hugging Face PEFT.
3. Effectuer une inférence avec le modèle adapté.
```

In [1]:
# Étape 1 : Installation et mise à jour des bibliothèques
# On force la mise à jour de torchao pour éviter l'ImportError dans peft
%pip install -U peft datasets transformers accelerate torchao

import os
# Création d'un dossier de cache pour stocker les résultats
if not os.path.exists('cache'):
    os.makedirs('cache')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


```markdown
### Étape 2 : Chargement du modèle et du dataset
Nous utilisons `bloomz-560m`, un modèle capable de comprendre plusieurs langues, et le jeu de données `english_quotes`.
```

In [2]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration des noms
model_name = "bigscience/bloomz-560m"

# Chargement du tokenizer (convertit le texte en nombres pour le modèle)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chargement du modèle de base (Foundation Model)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Chargement du dataset et échantillonnage de 10%
data = load_dataset("Abirate/english_quotes", split="train")
data = data.train_test_split(test_size=0.1, seed=42)["test"] # On prend 10% pour l'exercice

# Prétraitement : Tokenisation des citations
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

# Sélection de quelques exemples pour l'entraînement rapide du challenge
train_sample = data.select(range(10))
display(train_sample)

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 10
})

```markdown
### Étape 3 : Configuration de LoRA
LoRA ajoute de petites matrices de rang inférieur au modèle, réduisant considérablement le nombre de paramètres à entraîner.
```

In [3]:
import peft
from peft import LoraConfig, get_peft_model

# Configuration de LoRA
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Application de la configuration LoRA au modèle de base (foundation_model)
peft_model = get_peft_model(foundation_model, lora_config)

# Affichage des paramètres entraînables
print("Configuration LoRA terminée.")
peft_model.print_trainable_parameters()

Configuration LoRA terminée.
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176


```markdown
### Étape 4 : Entraînement du modèle
Nous configurons les arguments d'entraînement et lançons le `Trainer`.
```

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = "./cache/peft_lab_outputs"

# Arguments d'entraînement pour CPU
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True,
    logging_steps=1
)

# Initialisation du Trainer avec le modèle LoRA
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# Lancement de l'entraînement
print("Début de l'entraînement...")
trainer.train()

Début de l'entraînement...


```markdown
### Étape 5 : Sauvegarde et Inférence
Nous sauvegardons les poids appris par LoRA (l'adapteur) et testons la génération de texte.
```

In [ ]:
import time
import os

# S'assurer que les variables nécessaires sont définies
output_directory = "./cache/peft_lab_outputs"

# Sauvegarde du modèle finetuné
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print(f"Modèle sauvegardé dans : {peft_model_path}")

# Préparation de l'entrée pour le test
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Génération de texte
outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=20,
    no_repeat_ngram_size=2
)

# Décodage et affichage du résultat
print("--- Résultat de la génération ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])